In [1]:
# ============================================================
# TB PORTALS MARCH 2025
# NOTEBOOK 06
# CXR EXTRACTION AND TECHNICAL PREPROCESSING
# ============================================================

from pathlib import Path
import io
import os
import re
import json
import hashlib
import zipfile
import warnings

import numpy as np
import pandas as pd

try:
    import pydicom
    from pydicom.pixel_data_handlers.util import apply_voi_lut
except ImportError as e:
    raise ImportError(
        "pydicom is required. Install it once with:\n"
        "pip install pydicom"
    ) from e

print("Notebook 06 imports: PASS")
print("pydicom version:", pydicom.__version__)

Notebook 06 imports: PASS
pydicom version: 3.0.2


In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = Path(r"C:\TBP")
METADATA_DIR = BASE_DIR / "Metadata"

FINAL_COHORT_DIR = (
    METADATA_DIR /
    "Step_3B_6_50_Final_Temporal_Multimodal_Pair_Selection"
)

FINAL_COHORT_FILE = (
    FINAL_COHORT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Final_Prediction_Time_Multimodal_Cohort.csv"
)

MANIFEST_FILE = (
    METADATA_DIR /
    "TB_Portals_CXR_Manifest.csv"
)

# If your manifest has a different exact filename,
# the discovery cell below will locate it automatically.

OUTPUT_DIR = (
    METADATA_DIR /
    "Step_3B_6_51_CXR_Extraction_Technical_Preprocessing"
)

EXTRACTED_DIR = OUTPUT_DIR / "Extracted_DICOM"
PNG_DIR = OUTPUT_DIR / "Technical_PNG"
QC_DIR = OUTPUT_DIR / "QC"
PREVIEW_DIR = QC_DIR / "Preview"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
PNG_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Path configuration: PASS")

Output directory: C:\TBP\Metadata\Step_3B_6_51_CXR_Extraction_Technical_Preprocessing
Path configuration: PASS


In [3]:
# ============================================================
# LOCATE FINAL LOCKED COHORT
# ============================================================

if not FINAL_COHORT_FILE.exists():
    raise FileNotFoundError(
        f"Locked final cohort not found:\n{FINAL_COHORT_FILE}"
    )

cohort = pd.read_csv(FINAL_COHORT_FILE)

print("Final cohort loaded.")
print("Rows:", len(cohort))
print("Columns:", len(cohort.columns))

if len(cohort) != 2081:
    raise ValueError(
        f"Expected exactly 2081 final prediction-time conditions, "
        f"but found {len(cohort)}."
    )

print("Locked cohort row count: PASS")

Final cohort loaded.
Rows: 2081
Columns: 18
Locked cohort row count: PASS


In [4]:
# ============================================================
# COHORT SCHEMA VALIDATION
# ============================================================

print("\nFINAL COHORT COLUMNS")
print("=" * 80)

for i, col in enumerate(cohort.columns, start=1):
    print(f"{i:3d}. {col}")

required_candidates = {
    "condition_id": ["condition_id"],
    "cxr_url": ["series_instance_content_url"],
    "specimen_id": ["specimen_id"],
}

missing = []

for logical_name, candidates in required_candidates.items():
    if not any(c in cohort.columns for c in candidates):
        missing.append(
            f"{logical_name}: expected one of {candidates}"
        )

if missing:
    raise ValueError(
        "Required cohort fields are missing:\n" +
        "\n".join(missing)
    )

print("\nRequired cohort schema: PASS")


FINAL COHORT COLUMNS
  1. condition_id
  2. series_instance_content_url
  3. specimen_id
  4. imaging_date_num
  5. specimen_collection_date_num
  6. signed_temporal_delta_days
  7. absolute_temporal_distance_days
  8. prediction_temporal_class
  9. target_class
 10. target_binary
 11. cxr_resistance
 12. genomic_resistance
 13. cxr_record_count
 14. cxr_url_count
 15. multiple_cxr
 16. genomic_record_count
 17. genomic_drug_resistance_nunique
 18. human_review_excluded

Required cohort schema: PASS


In [5]:
# ============================================================
# IDENTIFY CXR URL FIELD
# ============================================================

CXR_URL_COL = "series_instance_content_url"

CONDITION_COL = "condition_id"
SPECIMEN_COL = "specimen_id"

if CXR_URL_COL not in cohort.columns:
    raise ValueError(
        f"Expected CXR URL column '{CXR_URL_COL}' not found."
    )

if CONDITION_COL not in cohort.columns:
    raise ValueError(
        "condition_id is required for condition-level integrity."
    )

if SPECIMEN_COL not in cohort.columns:
    raise ValueError(
        "specimen_id is required for multimodal pair integrity."
    )

cohort[CXR_URL_COL] = cohort[CXR_URL_COL].astype(str).str.strip()
cohort[CONDITION_COL] = cohort[CONDITION_COL].astype(str).str.strip()
cohort[SPECIMEN_COL] = cohort[SPECIMEN_COL].astype(str).str.strip()

if cohort[CXR_URL_COL].eq("").any():
    raise ValueError("Blank CXR URL detected.")

if cohort[CONDITION_COL].eq("").any():
    raise ValueError("Blank condition_id detected.")

if cohort[SPECIMEN_COL].eq("").any():
    raise ValueError("Blank specimen_id detected.")

if cohort[CXR_URL_COL].duplicated().any():
    dup_count = cohort[CXR_URL_COL].duplicated().sum()
    raise ValueError(
        f"Duplicate CXR URLs detected in final cohort: {dup_count}"
    )

print("CXR URL uniqueness: PASS")
print("Condition IDs:", cohort[CONDITION_COL].nunique())
print("Unique CXR URLs:", cohort[CXR_URL_COL].nunique())
print("Unique specimens:", cohort[SPECIMEN_COL].nunique())

CXR URL uniqueness: PASS
Condition IDs: 2081
Unique CXR URLs: 2081
Unique specimens: 2081


In [6]:
# ============================================================
# MANIFEST DISCOVERY
# ============================================================

if MANIFEST_FILE.exists():
    manifest_candidates = [MANIFEST_FILE]
else:
    manifest_candidates = list(METADATA_DIR.rglob("*.csv"))

manifest_matches = []

for f in manifest_candidates:
    try:
        cols = pd.read_csv(f, nrows=3).columns.tolist()
    except Exception:
        continue

    normalized = {str(c).strip().lower() for c in cols}

    if {"zip_file", "file"}.issubset(normalized):
        manifest_matches.append(f)

if len(manifest_matches) == 0:
    raise FileNotFoundError(
        "No manifest containing both 'zip_file' and 'file' "
        "could be located under C:\\TBP\\Metadata."
    )

print("Manifest candidates:")
for f in manifest_matches:
    print(" -", f)

if len(manifest_matches) > 1:
    print(
        "\nMultiple valid manifests found. "
        "The first candidate will NOT be silently selected."
    )
    
    # Prefer the explicitly configured path if available.
    if MANIFEST_FILE.exists():
        MANIFEST_FILE = MANIFEST_FILE
    else:
        raise RuntimeError(
            "Multiple valid manifests exist. "
            "Set MANIFEST_FILE explicitly in Cell 2."
        )
else:
    MANIFEST_FILE = manifest_matches[0]

print("\nSelected manifest:")
print(MANIFEST_FILE)

Manifest candidates:
 - C:\TBP\Metadata\TB_Portals_CXRs_March_2025_manifest.csv

Selected manifest:
C:\TBP\Metadata\TB_Portals_CXRs_March_2025_manifest.csv


In [7]:
# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(MANIFEST_FILE)

manifest.columns = [
    str(c).strip()
    for c in manifest.columns
]

required_manifest_cols = {"zip_file", "file"}

if not required_manifest_cols.issubset(manifest.columns):
    raise ValueError(
        f"Manifest must contain {required_manifest_cols}. "
        f"Found: {manifest.columns.tolist()}"
    )

manifest["zip_file"] = manifest["zip_file"].astype(str).str.strip()
manifest["file"] = manifest["file"].astype(str).str.strip()

manifest = manifest[
    manifest["zip_file"].ne("") &
    manifest["file"].ne("")
].copy()

print("Manifest rows:", len(manifest))
print("Unique ZIP files:", manifest["zip_file"].nunique())
print("Unique members:", manifest["file"].nunique())

print("Manifest schema: PASS")

Manifest rows: 16723
Unique ZIP files: 18
Unique members: 16723
Manifest schema: PASS


In [8]:
# ============================================================
# PATH NORMALIZATION
# ============================================================

def normalize_path_string(value):
    """
    Normalize a TB Portals path/URL for comparison only.
    Does not modify source data.
    """
    if pd.isna(value):
        return ""

    s = str(value).strip()

    s = s.replace("\\", "/")

    # Remove URL scheme if present
    s = re.sub(r"^[a-zA-Z]+://", "", s)

    # Normalize repeated separators
    s = re.sub(r"/+", "/", s)

    # Remove leading separators
    s = s.lstrip("/")

    return s.lower()


manifest["file_norm"] = manifest["file"].map(normalize_path_string)
manifest["zip_norm"] = manifest["zip_file"].map(normalize_path_string)

cohort["cxr_url_norm"] = cohort[CXR_URL_COL].map(
    normalize_path_string
)

print("Path normalization: PASS")

Path normalization: PASS


In [9]:
# ============================================================
# MANIFEST LOOKUP INDEXES
# ============================================================

manifest_exact = {}

for idx, row in manifest.iterrows():
    key = row["file_norm"]

    if key not in manifest_exact:
        manifest_exact[key] = []

    manifest_exact[key].append(idx)

manifest_basename = {}

for idx, row in manifest.iterrows():
    base = Path(row["file_norm"]).name

    if base not in manifest_basename:
        manifest_basename[base] = []

    manifest_basename[base].append(idx)

print("Exact manifest keys:", len(manifest_exact))
print("Basename keys:", len(manifest_basename))
print("Manifest index construction: PASS")

Exact manifest keys: 16723
Basename keys: 16723
Manifest index construction: PASS


In [10]:
# ============================================================
# RESOLVE SELECTED CXRs AGAINST MANIFEST
# ============================================================

def resolve_manifest_entry(url_norm):
    """
    Resolution hierarchy:

    1. Exact normalized path
    2. Exact suffix match
    3. Unique basename match

    Ambiguous matches are rejected.
    """

    # --------------------------------------------------------
    # 1. Exact match
    # --------------------------------------------------------
    exact = manifest_exact.get(url_norm, [])

    if len(exact) == 1:
        return exact[0], "EXACT"

    if len(exact) > 1:
        return None, "AMBIGUOUS_EXACT"

    # --------------------------------------------------------
    # 2. Suffix match
    # --------------------------------------------------------
    suffix_matches = []

    for idx, row in manifest.iterrows():
        m = row["file_norm"]

        if url_norm.endswith(m) or m.endswith(url_norm):
            suffix_matches.append(idx)

    if len(suffix_matches) == 1:
        return suffix_matches[0], "SUFFIX"

    if len(suffix_matches) > 1:
        return None, "AMBIGUOUS_SUFFIX"

    # --------------------------------------------------------
    # 3. Unique basename
    # --------------------------------------------------------
    basename = Path(url_norm).name

    base_matches = manifest_basename.get(basename, [])

    if len(base_matches) == 1:
        return base_matches[0], "BASENAME"

    if len(base_matches) > 1:
        return None, "AMBIGUOUS_BASENAME"

    return None, "NOT_FOUND"


resolution_rows = []

for _, row in cohort.iterrows():

    idx, method = resolve_manifest_entry(
        row["cxr_url_norm"]
    )

    result = {
        "condition_id": row[CONDITION_COL],
        "specimen_id": row[SPECIMEN_COL],
        "series_instance_content_url": row[CXR_URL_COL],
        "manifest_index": idx,
        "resolution_method": method,
    }

    if idx is not None:
        mrow = manifest.loc[idx]

        result["zip_file"] = mrow["zip_file"]
        result["file"] = mrow["file"]

    else:
        result["zip_file"] = ""
        result["file"] = ""

    resolution_rows.append(result)

resolution_df = pd.DataFrame(resolution_rows)

resolution_df.to_csv(
    QC_DIR /
    "TB_Portals_March2025_Notebook06_CXR_Manifest_Resolution.csv",
    index=False
)

print(
    resolution_df["resolution_method"]
    .value_counts(dropna=False)
)

resolution_method
EXACT    2081
Name: count, dtype: int64


In [11]:
# ============================================================
# MANIFEST RESOLUTION GATE
# ============================================================

bad_resolution = resolution_df[
    ~resolution_df["resolution_method"].isin(
        ["EXACT", "SUFFIX", "BASENAME"]
    )
].copy()

print("Total selected CXRs:", len(resolution_df))
print("Successfully resolved:", len(resolution_df) - len(bad_resolution))
print("Unresolved/ambiguous:", len(bad_resolution))

if len(bad_resolution) > 0:

    bad_resolution.to_csv(
        QC_DIR /
        "Notebook06_Unresolved_or_Ambiguous_CXR_Mappings.csv",
        index=False
    )

    raise RuntimeError(
        f"{len(bad_resolution)} selected CXR files could not be "
        "uniquely resolved against the TB Portals manifest. "
        "No image extraction was started."
    )

print("Manifest resolution: PASS")

Total selected CXRs: 2081
Successfully resolved: 2081
Unresolved/ambiguous: 0
Manifest resolution: PASS


In [12]:
# ============================================================
# LOCATE ZIP ARCHIVES
# ============================================================

print("Searching for ZIP archives under C:\\TBP ...")

zip_files = list(BASE_DIR.rglob("*.zip"))

print("ZIP archives discovered:", len(zip_files))

if len(zip_files) == 0:
    raise FileNotFoundError(
        "No ZIP archives were found under C:\\TBP."
    )

zip_by_name = {}

for z in zip_files:

    name_norm = normalize_path_string(z.name)

    zip_by_name.setdefault(name_norm, []).append(z)

print("ZIP filename index created.")

Searching for ZIP archives under C:\TBP ...
ZIP archives discovered: 18
ZIP filename index created.


In [13]:
# ============================================================
# RESOLVE ZIP FILES
# ============================================================

def resolve_zip(zip_name):
    """
    Resolve manifest ZIP name to an actual local ZIP.
    Reject ambiguity.
    """

    norm = normalize_path_string(zip_name)
    basename = Path(norm).name

    candidates = zip_by_name.get(basename, [])

    if len(candidates) == 1:
        return candidates[0], "EXACT_FILENAME"

    if len(candidates) > 1:
        return None, "AMBIGUOUS_ZIP"

    # fallback: compare normalized full path suffix
    suffix_candidates = []

    for z in zip_files:

        z_norm = normalize_path_string(str(z))

        if z_norm.endswith(norm):
            suffix_candidates.append(z)

    if len(suffix_candidates) == 1:
        return suffix_candidates[0], "PATH_SUFFIX"

    if len(suffix_candidates) > 1:
        return None, "AMBIGUOUS_ZIP"

    return None, "ZIP_NOT_FOUND"


zip_resolution = []

for zip_name in resolution_df["zip_file"].drop_duplicates():

    zpath, method = resolve_zip(zip_name)

    zip_resolution.append({
        "manifest_zip_file": zip_name,
        "local_zip_path": str(zpath) if zpath else "",
        "zip_resolution_method": method
    })

zip_resolution_df = pd.DataFrame(zip_resolution)

zip_resolution_df.to_csv(
    QC_DIR /
    "TB_Portals_March2025_Notebook06_ZIP_Resolution.csv",
    index=False
)

print(
    zip_resolution_df["zip_resolution_method"]
    .value_counts(dropna=False)
)

zip_resolution_method
EXACT_FILENAME    18
Name: count, dtype: int64


In [14]:
# ============================================================
# ZIP RESOLUTION GATE
# ============================================================

bad_zips = zip_resolution_df[
    ~zip_resolution_df["zip_resolution_method"].isin(
        ["EXACT_FILENAME", "PATH_SUFFIX"]
    )
]

if len(bad_zips) > 0:

    bad_zips.to_csv(
        QC_DIR /
        "Notebook06_Unresolved_ZIP_Archives.csv",
        index=False
    )

    raise RuntimeError(
        f"{len(bad_zips)} ZIP archives could not be uniquely resolved."
    )

print("ZIP archive resolution: PASS")

ZIP archive resolution: PASS


In [16]:
# ============================================================
# CELL 15 — BUILD FINAL EXTRACTION MAP
# ============================================================

# The ZIP-resolution table uses:
#   manifest_zip_file
# while resolution_df uses:
#   zip_file
#
# Rename explicitly before merging so the key is identical.

if "zip_file" not in resolution_df.columns:
    raise RuntimeError(
        "resolution_df does not contain 'zip_file'. "
        f"Available columns: {resolution_df.columns.tolist()}"
    )

if "manifest_zip_file" not in zip_resolution_df.columns:
    raise RuntimeError(
        "zip_resolution_df does not contain 'manifest_zip_file'. "
        f"Available columns: {zip_resolution_df.columns.tolist()}"
    )

zip_resolution_for_merge = zip_resolution_df.rename(
    columns={
        "manifest_zip_file": "zip_file"
    }
).copy()

# Confirm one unique local ZIP path per manifest ZIP name.
if zip_resolution_for_merge["zip_file"].duplicated().any():
    duplicate_zip_names = (
        zip_resolution_for_merge.loc[
            zip_resolution_for_merge["zip_file"].duplicated(
                keep=False
            ),
            "zip_file"
        ]
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "ZIP resolution table contains duplicate manifest ZIP names:\n"
        + "\n".join(map(str, duplicate_zip_names))
    )

resolution_df = resolution_df.merge(
    zip_resolution_for_merge[
        [
            "zip_file",
            "local_zip_path",
            "zip_resolution_method"
        ]
    ],
    on="zip_file",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Validate the merge
# ------------------------------------------------------------

if len(resolution_df) != 2081:
    raise RuntimeError(
        f"Expected 2081 rows after ZIP merge, "
        f"found {len(resolution_df)}."
    )

if resolution_df["local_zip_path"].isna().any():
    missing_count = int(
        resolution_df["local_zip_path"].isna().sum()
    )

    raise RuntimeError(
        f"{missing_count} selected CXR records have no "
        "resolved local ZIP path."
    )

if resolution_df["local_zip_path"].eq("").any():
    missing_count = int(
        resolution_df["local_zip_path"].eq("").sum()
    )

    raise RuntimeError(
        f"{missing_count} selected CXR records have a blank "
        "local ZIP path."
    )

print("Extraction map rows:", len(resolution_df))
print(
    "Unique resolved ZIP archives:",
    resolution_df["local_zip_path"].nunique()
)
print("Final extraction map: PASS")

Extraction map rows: 2081
Unique resolved ZIP archives: 18
Final extraction map: PASS


In [17]:
# ============================================================
# CELL 16 — VERIFY ZIP MEMBERS
# ============================================================

member_verification = []

zip_groups = resolution_df.groupby(
    "local_zip_path",
    sort=True
)

for zip_path_str, group in zip_groups:

    zip_path = Path(zip_path_str)

    if not zip_path.exists():
        raise FileNotFoundError(
            f"ZIP archive does not exist:\n{zip_path}"
        )

    print(
        f"Checking ZIP: {zip_path.name} "
        f"({len(group)} selected members)"
    )

    with zipfile.ZipFile(zip_path, "r") as z:

        # Preserve original member names while creating a
        # normalized lookup.
        members = {}

        for original_name in z.namelist():

            normalized_name = normalize_path_string(
                original_name
            )

            if normalized_name in members:
                # Multiple ZIP entries normalize to the same
                # path. We must not silently choose one.
                members[normalized_name].append(
                    original_name
                )
            else:
                members[normalized_name] = [
                    original_name
                ]

        for _, row in group.iterrows():

            target = normalize_path_string(
                row["file"]
            )

            candidates = members.get(
                target,
                []
            )

            if len(candidates) == 1:

                actual_member = candidates[0]
                status = "FOUND"

            elif len(candidates) > 1:

                actual_member = ""
                status = "AMBIGUOUS_MEMBER"

            else:

                # Controlled suffix fallback.
                suffix_candidates = []

                for normalized_name, originals in members.items():

                    if (
                        normalized_name.endswith(target)
                        or
                        target.endswith(normalized_name)
                    ):
                        suffix_candidates.extend(
                            originals
                        )

                if len(suffix_candidates) == 1:

                    actual_member = suffix_candidates[0]
                    status = "FOUND_SUFFIX"

                elif len(suffix_candidates) > 1:

                    actual_member = ""
                    status = "AMBIGUOUS_SUFFIX"

                else:

                    actual_member = ""
                    status = "MEMBER_NOT_FOUND"

            member_verification.append({
                "condition_id":
                    row["condition_id"],

                "series_instance_content_url":
                    row["series_instance_content_url"],

                "zip_path":
                    str(zip_path),

                "manifest_member":
                    row["file"],

                "actual_member":
                    actual_member,

                "member_status":
                    status
            })

member_verification_df = pd.DataFrame(
    member_verification
)

member_verification_df.to_csv(
    QC_DIR /
    "TB_Portals_March2025_Notebook06_ZIP_Member_Verification.csv",
    index=False
)

print("\nZIP member verification:")
print(
    member_verification_df[
        "member_status"
    ].value_counts(dropna=False)
)

Checking ZIP: TB_Portals_CXRs_March_2025_01_of_18.zip (112 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_02_of_18.zip (125 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_03_of_18.zip (107 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_04_of_18.zip (112 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_05_of_18.zip (112 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_06_of_18.zip (91 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_07_of_18.zip (139 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_08_of_18.zip (131 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_09_of_18.zip (116 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_10_of_18.zip (109 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_11_of_18.zip (113 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_12_of_18.zip (124 selected members)
Checking ZIP: TB_Portals_CXRs_March_2025_13_of_18.zip (123 selected members)


In [18]:
# ============================================================
# CELL 17 — ZIP MEMBER VERIFICATION GATE
# ============================================================

allowed_member_status = {
    "FOUND",
    "FOUND_SUFFIX"
}

bad_members = member_verification_df[
    ~member_verification_df[
        "member_status"
    ].isin(allowed_member_status)
].copy()

print(
    "Total selected CXR members:",
    len(member_verification_df)
)

print(
    "Successfully verified:",
    len(member_verification_df) - len(bad_members)
)

print(
    "Unresolved / ambiguous:",
    len(bad_members)
)

if len(bad_members) > 0:

    bad_members.to_csv(
        QC_DIR /
        "Notebook06_Missing_or_Ambiguous_ZIP_Members.csv",
        index=False
    )

    raise RuntimeError(
        f"{len(bad_members)} selected CXR members could not "
        "be uniquely located inside their ZIP archives. "
        "No DICOM extraction was started."
    )

if len(member_verification_df) != 2081:
    raise RuntimeError(
        "ZIP member verification does not contain exactly "
        "2081 selected records."
    )

print("All selected ZIP members verified: PASS")

Total selected CXR members: 2081
Successfully verified: 2081
Unresolved / ambiguous: 0
All selected ZIP members verified: PASS


In [19]:
# ============================================================
# CELL 18 — BUILD FINAL EXTRACTION TABLE
# ============================================================

member_lookup = member_verification_df[
    [
        "condition_id",
        "series_instance_content_url",
        "actual_member",
        "member_status"
    ]
].copy()

# The pair condition_id + CXR URL must be unique.
if member_lookup.duplicated(
    subset=[
        "condition_id",
        "series_instance_content_url"
    ]
).any():

    raise RuntimeError(
        "Duplicate condition_id + CXR URL combinations "
        "found in ZIP member verification."
    )

extraction_map = resolution_df.merge(
    member_lookup,
    on=[
        "condition_id",
        "series_instance_content_url"
    ],
    how="left",
    validate="one_to_one"
)

if len(extraction_map) != 2081:
    raise RuntimeError(
        f"Expected 2081 extraction records, "
        f"found {len(extraction_map)}."
    )

required_extraction_columns = [
    "condition_id",
    "specimen_id",
    "series_instance_content_url",
    "zip_file",
    "local_zip_path",
    "file",
    "actual_member",
    "member_status"
]

missing = [
    c for c in required_extraction_columns
    if c not in extraction_map.columns
]

if missing:
    raise RuntimeError(
        "Extraction map is missing required columns:\n"
        + "\n".join(missing)
    )

if extraction_map["actual_member"].eq("").any():
    raise RuntimeError(
        "Blank ZIP member found in final extraction map."
    )

print("Extraction map rows:", len(extraction_map))
print("Extraction map schema: PASS")

Extraction map rows: 2081
Extraction map schema: PASS


In [20]:
# ============================================================
# CELL 19 — SAFE OUTPUT IDENTIFIERS
# ============================================================

def safe_token(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value[:150]


def make_output_stem(row):
    condition = safe_token(row["condition_id"])

    url_hash = hashlib.sha256(
        str(row["series_instance_content_url"])
        .encode("utf-8")
    ).hexdigest()[:12]

    return f"{condition}_{url_hash}"


extraction_map["output_stem"] = extraction_map.apply(
    make_output_stem,
    axis=1
)

if extraction_map["output_stem"].duplicated().any():
    raise RuntimeError(
        "Output filename collision detected."
    )

print(
    "Unique output identifiers:",
    extraction_map["output_stem"].nunique()
)

print("Output naming integrity: PASS")

Unique output identifiers: 2081
Output naming integrity: PASS


In [21]:
# ============================================================
# CELL 20 — DICOM TECHNICAL CONVERSION FUNCTIONS
# ============================================================

from PIL import Image

def dicom_to_float_image(ds):

    arr = ds.pixel_array

    photometric = str(
        getattr(
            ds,
            "PhotometricInterpretation",
            ""
        )
    ).upper()

    # --------------------------------------------------------
    # COLOR
    # --------------------------------------------------------

    if photometric in {
        "RGB",
        "YBR_FULL",
        "YBR_FULL_422",
        "YBR_ICT",
        "YBR_RCT"
    }:

        arr = arr.astype(np.float32)

        if arr.ndim != 3 or arr.shape[-1] not in {3, 4}:
            raise ValueError(
                f"Unexpected color image shape: {arr.shape}"
            )

        if arr.shape[-1] == 4:
            arr = arr[..., :3]

        return arr, "COLOR"

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    arr = arr.astype(np.float32)

    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    arr = arr * slope + intercept

    # --------------------------------------------------------
    # MONOCHROME1
    # --------------------------------------------------------

    if photometric == "MONOCHROME1":

        finite = np.isfinite(arr)

        if not finite.any():
            raise ValueError(
                "MONOCHROME1 image contains no finite pixels."
            )

        lo = np.nanmin(arr)
        hi = np.nanmax(arr)

        arr = hi + lo - arr

    # --------------------------------------------------------
    # OPTIONAL VOI LUT / WINDOW
    # --------------------------------------------------------

    has_window = (
        hasattr(ds, "WindowCenter")
        and
        hasattr(ds, "WindowWidth")
    )

    has_voi_lut = hasattr(
        ds,
        "VOILUTSequence"
    )

    if has_voi_lut or has_window:

        try:

            voi = apply_voi_lut(
                ds.pixel_array,
                ds
            ).astype(np.float32)

            if photometric == "MONOCHROME1":

                lo = np.nanmin(voi)
                hi = np.nanmax(voi)

                voi = hi + lo - voi

            arr = voi

        except Exception:
            # Preserve the rescaled pixel representation if
            # optional VOI processing cannot be applied.
            pass

    return arr, "GRAYSCALE"


def normalize_to_uint16(arr):

    arr = np.asarray(
        arr,
        dtype=np.float32
    )

    if not np.isfinite(arr).any():
        raise ValueError(
            "Image contains no finite pixels."
        )

    valid = arr[np.isfinite(arr)]

    lo = float(
        np.percentile(
            valid,
            0.5
        )
    )

    hi = float(
        np.percentile(
            valid,
            99.5
        )
    )

    if hi <= lo:
        raise ValueError(
            f"Degenerate intensity range: "
            f"low={lo}, high={hi}"
        )

    normalized = np.clip(
        (arr - lo) / (hi - lo),
        0,
        1
    )

    normalized = np.nan_to_num(
        normalized,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    output = np.round(
        normalized * 65535
    ).astype(np.uint16)

    return output, lo, hi


print("DICOM conversion functions: PASS")

DICOM conversion functions: PASS


In [30]:
# ============================================================
# CELL 21 — RESUMABLE CXR EXTRACTION AND TECHNICAL PROCESSING
#              WITH NONE-SAFE DICOM METADATA HANDLING
# ============================================================

qc_rows = []

total_records = len(extraction_map)

print("=" * 80)
print("RESUMABLE CXR EXTRACTION")
print("=" * 80)
print("Total records:", total_records)
print("PNG directory:", PNG_DIR)
print("DICOM directory:", EXTRACTED_DIR)
print()


# ============================================================
# HELPER — SAFE FLOAT CONVERSION
# ============================================================

def safe_float(value, default):
    """
    Convert a DICOM numeric value safely to float.

    Handles:
    - None
    - empty strings
    - invalid values

    Important:
    We do NOT alter the pixel data because of missing
    optional metadata. We only use the default value
    for the QC metadata field.
    """

    if value is None:
        return float(default)

    try:
        return float(value)

    except (TypeError, ValueError):
        return float(default)


# ============================================================
# HELPER — SAFE INT CONVERSION
# ============================================================

def safe_int(value, default):
    """
    Convert a DICOM numeric value safely to int.
    """

    if value is None:
        return int(default)

    try:
        return int(value)

    except (TypeError, ValueError):
        return int(default)


# ============================================================
# MAIN EXTRACTION LOOP
# ============================================================

for position, (i, row) in enumerate(
    extraction_map.iterrows(),
    start=1
):

    condition_id = row["condition_id"]

    specimen_id = row["specimen_id"]

    zip_path = Path(
        row["local_zip_path"]
    )

    member = row["actual_member"]

    stem = row["output_stem"]

    dicom_path = (
        EXTRACTED_DIR /
        f"{stem}.dcm"
    )

    png_path = (
        PNG_DIR /
        f"{stem}.png"
    )

    print(
        f"[{position:04d}/{total_records}] "
        f"{condition_id}"
    )

    record = {

        "condition_id":
            condition_id,

        "specimen_id":
            specimen_id,

        "series_instance_content_url":
            row["series_instance_content_url"],

        "zip_file":
            row["zip_file"],

        "zip_member":
            member,

        "status":
            "ERROR",

        "dicom_path":
            str(dicom_path),

        "png_path":
            str(png_path)
    }


    try:

        # ====================================================
        # STEP 1
        # Check whether both existing outputs are valid.
        # ====================================================

        existing_outputs = (

            dicom_path.exists()

            and

            png_path.exists()

            and

            dicom_path.stat().st_size > 0

            and

            png_path.stat().st_size > 0
        )


        if existing_outputs:

            try:

                # --------------------------------------------
                # Validate PNG
                # --------------------------------------------

                with Image.open(
                    png_path
                ) as existing_image:

                    existing_image.verify()


                # --------------------------------------------
                # Validate DICOM
                # --------------------------------------------

                existing_ds = pydicom.dcmread(
                    dicom_path,
                    force=False,
                    stop_before_pixels=False
                )


                # --------------------------------------------
                # Existing files are valid.
                # Reuse them.
                # --------------------------------------------

                record.update({

                    "status":
                        "REUSED",

                    "dicom_bytes":
                        dicom_path.stat().st_size,

                    "dicom_sha256":
                        hashlib.sha256(
                            dicom_path.read_bytes()
                        ).hexdigest(),

                    "png_sha256":
                        hashlib.sha256(
                            png_path.read_bytes()
                        ).hexdigest(),

                    "dicom_modality":
                        str(
                            getattr(
                                existing_ds,
                                "Modality",
                                ""
                            )
                        ),

                    "photometric_interpretation":
                        str(
                            getattr(
                                existing_ds,
                                "PhotometricInterpretation",
                                ""
                            )
                        ),

                    "rows":
                        safe_int(
                            getattr(
                                existing_ds,
                                "Rows",
                                None
                            ),
                            -1
                        ),

                    "columns":
                        safe_int(
                            getattr(
                                existing_ds,
                                "Columns",
                                None
                            ),
                            -1
                        ),

                    "samples_per_pixel":
                        safe_int(
                            getattr(
                                existing_ds,
                                "SamplesPerPixel",
                                None
                            ),
                            -1
                        ),

                    "bits_allocated":
                        safe_int(
                            getattr(
                                existing_ds,
                                "BitsAllocated",
                                None
                            ),
                            -1
                        ),

                    "bits_stored":
                        safe_int(
                            getattr(
                                existing_ds,
                                "BitsStored",
                                None
                            ),
                            -1
                        ),

                    "high_bit":
                        safe_int(
                            getattr(
                                existing_ds,
                                "HighBit",
                                None
                            ),
                            -1
                        ),

                    "number_of_frames":
                        safe_int(
                            getattr(
                                existing_ds,
                                "NumberOfFrames",
                                None
                            ),
                            1
                        ),

                    "planar_configuration":
                        str(
                            getattr(
                                existing_ds,
                                "PlanarConfiguration",
                                ""
                            )
                        ),

                    "rescale_slope":
                        safe_float(
                            getattr(
                                existing_ds,
                                "RescaleSlope",
                                None
                            ),
                            1.0
                        ),

                    "rescale_intercept":
                        safe_float(
                            getattr(
                                existing_ds,
                                "RescaleIntercept",
                                None
                            ),
                            0.0
                        ),

                    "pixel_spacing":
                        str(
                            getattr(
                                existing_ds,
                                "PixelSpacing",
                                None
                            )
                        ),

                    "image_type":
                        "REUSED_EXISTING_OUTPUT"
                })


                print(
                    "    -> REUSED existing valid outputs"
                )


                qc_rows.append(
                    record
                )

                continue


            except Exception:

                # Existing output failed validation.
                # Reprocess it from the original ZIP.

                print(
                    "    -> Existing output failed validation."
                )

                print(
                    "    -> Reprocessing record."
                )


        # ====================================================
        # STEP 2
        # Verify source ZIP exists.
        # ====================================================

        if not zip_path.exists():

            raise FileNotFoundError(
                f"ZIP file not found: {zip_path}"
            )


        # ====================================================
        # STEP 3
        # Read original DICOM bytes directly from ZIP.
        # ====================================================

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as z:

            names = z.namelist()

            if member not in names:

                raise FileNotFoundError(
                    f"ZIP member not found: {member}"
                )

            raw_bytes = z.read(
                member
            )


        if len(raw_bytes) == 0:

            raise ValueError(
                "Extracted DICOM bytes are empty."
            )


        # ====================================================
        # STEP 4
        # Preserve exact original DICOM.
        # ====================================================

        dicom_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with open(
            dicom_path,
            "wb"
        ) as f:

            f.write(
                raw_bytes
            )


        # ====================================================
        # STEP 5
        # Read DICOM.
        # ====================================================

        ds = pydicom.dcmread(
            io.BytesIO(raw_bytes),
            force=False
        )


        # ====================================================
        # STEP 6
        # Technical metadata.
        #
        # IMPORTANT:
        # All optional numeric metadata is now None-safe.
        # ====================================================

        rows = safe_int(
            getattr(
                ds,
                "Rows",
                None
            ),
            -1
        )

        cols = safe_int(
            getattr(
                ds,
                "Columns",
                None
            ),
            -1
        )

        photometric = str(
            getattr(
                ds,
                "PhotometricInterpretation",
                ""
            )
        )

        modality = str(
            getattr(
                ds,
                "Modality",
                ""
            )
        )

        bits_allocated = safe_int(
            getattr(
                ds,
                "BitsAllocated",
                None
            ),
            -1
        )

        bits_stored = safe_int(
            getattr(
                ds,
                "BitsStored",
                None
            ),
            -1
        )

        high_bit = safe_int(
            getattr(
                ds,
                "HighBit",
                None
            ),
            -1
        )

        samples_per_pixel = safe_int(
            getattr(
                ds,
                "SamplesPerPixel",
                None
            ),
            -1
        )

        planar_configuration = getattr(
            ds,
            "PlanarConfiguration",
            ""
        )

        number_of_frames = safe_int(
            getattr(
                ds,
                "NumberOfFrames",
                None
            ),
            1
        )

        slope = safe_float(
            getattr(
                ds,
                "RescaleSlope",
                None
            ),
            1.0
        )

        intercept = safe_float(
            getattr(
                ds,
                "RescaleIntercept",
                None
            ),
            0.0
        )

        pixel_spacing = getattr(
            ds,
            "PixelSpacing",
            None
        )


        # ====================================================
        # STEP 7
        # Validate mandatory image dimensions.
        # ====================================================

        if rows <= 0:

            raise ValueError(
                f"Invalid DICOM Rows value: {rows}"
            )

        if cols <= 0:

            raise ValueError(
                f"Invalid DICOM Columns value: {cols}"
            )


        # ====================================================
        # STEP 8
        # Reject multi-frame images.
        # ====================================================

        if number_of_frames != 1:

            raise ValueError(
                f"Multi-frame image: "
                f"{number_of_frames} frames"
            )


        # ====================================================
        # STEP 9
        # Decode pixel data.
        #
        # We keep the existing project function unchanged.
        # ====================================================

        arr, image_type = (
            dicom_to_float_image(ds)
        )


        # ====================================================
        # STEP 10
        # Validate decoded array.
        # ====================================================

        if arr.size == 0:

            raise ValueError(
                "Decoded pixel array is empty."
            )

        if not np.isfinite(arr).any():

            raise ValueError(
                "Decoded pixel array contains no finite values."
            )


        # ====================================================
        # STEP 11
        # Technical 16-bit PNG representation.
        # ====================================================

        png_array, clip_lo, clip_hi = (
            normalize_to_uint16(arr)
        )


        png_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )


        if png_array.ndim == 2:

            image = Image.fromarray(
                png_array,
                mode="I;16"
            )


        elif (

            png_array.ndim == 3

            and

            png_array.shape[-1] == 3

        ):

            rgb8 = (
                png_array / 257.0
            ).clip(
                0,
                255
            ).astype(
                np.uint8
            )

            image = Image.fromarray(
                rgb8,
                mode="RGB"
            )


        else:

            raise ValueError(
                f"Unsupported processed image shape: "
                f"{png_array.shape}"
            )


        # ====================================================
        # STEP 12
        # Save PNG.
        # ====================================================

        image.save(
            png_path,
            format="PNG"
        )


        # ====================================================
        # STEP 13
        # Verify PNG after writing.
        # ====================================================

        if not png_path.exists():

            raise IOError(
                "PNG file was not created."
            )

        if png_path.stat().st_size == 0:

            raise IOError(
                "PNG file was created but is empty."
            )

        with Image.open(
            png_path
        ) as check_image:

            check_image.verify()


        # ====================================================
        # STEP 14
        # Hash outputs.
        # ====================================================

        dicom_sha256 = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        png_sha256 = hashlib.sha256(
            png_path.read_bytes()
        ).hexdigest()


        # ====================================================
        # STEP 15
        # Store QC record.
        # ====================================================

        record.update({

            "status":
                "PASS",

            "dicom_bytes":
                len(raw_bytes),

            "dicom_sha256":
                dicom_sha256,

            "dicom_modality":
                modality,

            "photometric_interpretation":
                photometric,

            "rows":
                rows,

            "columns":
                cols,

            "samples_per_pixel":
                samples_per_pixel,

            "bits_allocated":
                bits_allocated,

            "bits_stored":
                bits_stored,

            "high_bit":
                high_bit,

            "number_of_frames":
                number_of_frames,

            "planar_configuration":
                planar_configuration,

            "rescale_slope":
                slope,

            "rescale_intercept":
                intercept,

            "pixel_spacing":
                str(pixel_spacing),

            "image_type":
                image_type,

            "native_array_shape":
                str(arr.shape),

            "native_min":
                float(
                    np.nanmin(arr)
                ),

            "native_max":
                float(
                    np.nanmax(arr)
                ),

            "native_mean":
                float(
                    np.nanmean(arr)
                ),

            "clip_percentile_low":
                clip_lo,

            "clip_percentile_high":
                clip_hi,

            "dicom_path":
                str(dicom_path),

            "png_path":
                str(png_path),

            "png_sha256":
                png_sha256
        })


        print(
            "    -> PASS"
        )


    except Exception as e:

        record.update({

            "status":
                "ERROR",

            "error_type":
                type(e).__name__,

            "error_message":
                str(e)
        })

        print(
            f"    -> ERROR: "
            f"{type(e).__name__}: {e}"
        )


    qc_rows.append(
        record
    )


# ============================================================
# STEP 16
# Create QC DataFrame.
# ============================================================

qc_df = pd.DataFrame(
    qc_rows
)


# ============================================================
# STEP 17
# Save QC report.
# ============================================================

qc_file = (
    QC_DIR /
    "TB_Portals_March2025_Notebook06_CXR_Technical_QC.csv"
)

qc_df.to_csv(
    qc_file,
    index=False
)


# ============================================================
# STEP 18
# Final summary.
# ============================================================

print("\n" + "=" * 80)
print("EXTRACTION RUN FINISHED")
print("=" * 80)

print(
    "\nStatus distribution:"
)

print(
    qc_df["status"].value_counts(
        dropna=False
    )
)

print(
    "\nQC rows:",
    len(qc_df)
)

print(
    "Expected rows:",
    total_records
)

print(
    "\nPNG files currently present:",
    len(
        list(
            PNG_DIR.glob("*.png")
        )
    )
)

print(
    "DICOM files currently present:",
    len(
        list(
            EXTRACTED_DIR.glob("*.dcm")
        )
    )
)

print(
    "\nQC file:",
    qc_file
)

print("=" * 80)

RESUMABLE CXR EXTRACTION
Total records: 2081
PNG directory: C:\TBP\Metadata\Step_3B_6_51_CXR_Extraction_Technical_Preprocessing\Technical_PNG
DICOM directory: C:\TBP\Metadata\Step_3B_6_51_CXR_Extraction_Technical_Preprocessing\Extracted_DICOM

[0001/2081] 0000f7e6-bbd2-468d-834d-f14948c5d902
    -> REUSED existing valid outputs
[0002/2081] 000e6c76-07b4-43cc-b668-048612911ce4
    -> REUSED existing valid outputs
[0003/2081] 001dc6ee-4925-4505-a4f6-e79d2e31d1cf
    -> REUSED existing valid outputs
[0004/2081] 00376922-57ad-47be-9b42-9fa35cfbb1a8
    -> REUSED existing valid outputs
[0005/2081] 003e095a-db07-4abe-b07d-3050fc77f925
    -> REUSED existing valid outputs
[0006/2081] 004e71c5-d99a-42ae-bfa3-c46ddca6aece
    -> REUSED existing valid outputs
[0007/2081] 00659068-bc90-463a-8911-7e827ba59361
    -> REUSED existing valid outputs
[0008/2081] 006f0ffc-40f1-4714-9ef4-cc3efb801d2a
    -> REUSED existing valid outputs
[0009/2081] 0073c77c-af2c-4a71-8a39-bf667b5ec936
    -> REUSED exist

C:\Users\Gobika\AppData\Local\Temp\ipykernel_5784\3541307882.py:632: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


    -> PASS
[0014/2081] 010b23c3-1775-4aa5-abf3-59f0fb326655
    -> REUSED existing valid outputs
[0015/2081] 01374588-24d4-445a-a01d-4be34e25dc75
    -> REUSED existing valid outputs
[0016/2081] 014e5607-00ea-4ff1-bcd5-15290bf02cf3
    -> REUSED existing valid outputs
[0017/2081] 0171707e-3310-403d-a675-5d309bcc9bcf
    -> REUSED existing valid outputs
[0018/2081] 01818375-d53f-4d08-a981-d6d13650c90f
    -> REUSED existing valid outputs
[0019/2081] 0191bec4-abc7-4f60-9d2a-c7f960baaecf
    -> REUSED existing valid outputs
[0020/2081] 019539cf-b2ad-4d59-9f54-18ee71cd0f46
    -> REUSED existing valid outputs
[0021/2081] 01b78c92-c8f6-4564-b43f-a3a774f3d9eb
    -> PASS
[0022/2081] 01f25903-2a8f-4bde-bad8-5e7e58310747
    -> REUSED existing valid outputs
[0023/2081] 0226b6b3-5917-4b2a-bf40-d7d9a7329510
    -> REUSED existing valid outputs
[0024/2081] 0237b7e6-0fce-4b3b-9b80-266333905166
    -> REUSED existing valid outputs
[0025/2081] 024f9918-1b40-4153-9fae-9aca85dde38b
    -> REUSED exis

In [26]:
# ============================================================
# CELL 21A — DIAGNOSTIC FOR THE 43 FAILED DICOM RECORDS
# ============================================================

import traceback
import io
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np
import pydicom


print("=" * 80)
print("DIAGNOSTIC: FAILED CXR EXTRACTION RECORDS")
print("=" * 80)


# ============================================================
# 1. Load the latest QC file
# ============================================================

qc_file = (
    QC_DIR /
    "TB_Portals_March2025_Notebook06_CXR_Technical_QC.csv"
)

qc_existing = pd.read_csv(
    qc_file
)

print("QC rows:", len(qc_existing))


# ============================================================
# 2. Select ONLY the failed records
# ============================================================

failed_df = qc_existing[
    qc_existing["status"].astype(str).str.upper() == "ERROR"
].copy()


print(
    "Failed records:",
    len(failed_df)
)

print()


if len(failed_df) == 0:

    print("No ERROR records found.")
    print("Do not continue with this diagnostic.")


else:

    print("Beginning detailed diagnostic...")
    print()


# ============================================================
# 3. Diagnostic output
# ============================================================

diagnostic_rows = []


# ============================================================
# 4. Inspect every failed record
# ============================================================

for position, (_, row) in enumerate(
    failed_df.iterrows(),
    start=1
):

    condition_id = row["condition_id"]

    specimen_id = row["specimen_id"]

    dicom_path = Path(
        row["dicom_path"]
    )

    print(
        f"[{position:02d}/{len(failed_df)}] "
        f"{condition_id}"
    )


    result = {

        "condition_id":
            condition_id,

        "specimen_id":
            specimen_id,

        "dicom_path":
            str(dicom_path),

        "dicom_exists":
            dicom_path.exists(),

        "dicom_size":
            (
                dicom_path.stat().st_size
                if dicom_path.exists()
                else None
            ),

        "rescale_slope_raw":
            None,

        "rescale_intercept_raw":
            None,

        "rows_raw":
            None,

        "columns_raw":
            None,

        "bits_allocated_raw":
            None,

        "bits_stored_raw":
            None,

        "high_bit_raw":
            None,

        "samples_per_pixel_raw":
            None,

        "number_of_frames_raw":
            None,

        "photometric_interpretation_raw":
            None,

        "modality_raw":
            None,

        "pixel_array_shape":
            None,

        "pixel_array_dtype":
            None,

        "dicom_to_float_image_status":
            None,

        "dicom_to_float_image_error":
            None,

        "full_traceback":
            None
    }


    try:

        # ----------------------------------------------------
        # Read the preserved DICOM
        # ----------------------------------------------------

        if not dicom_path.exists():

            raise FileNotFoundError(
                f"DICOM does not exist: {dicom_path}"
            )


        ds = pydicom.dcmread(
            dicom_path,
            force=False
        )


        # ----------------------------------------------------
        # Capture RAW DICOM values
        # ----------------------------------------------------

        slope = getattr(
            ds,
            "RescaleSlope",
            "<MISSING>"
        )

        intercept = getattr(
            ds,
            "RescaleIntercept",
            "<MISSING>"
        )

        rows = getattr(
            ds,
            "Rows",
            "<MISSING>"
        )

        columns = getattr(
            ds,
            "Columns",
            "<MISSING>"
        )

        bits_allocated = getattr(
            ds,
            "BitsAllocated",
            "<MISSING>"
        )

        bits_stored = getattr(
            ds,
            "BitsStored",
            "<MISSING>"
        )

        high_bit = getattr(
            ds,
            "HighBit",
            "<MISSING>"
        )

        samples_per_pixel = getattr(
            ds,
            "SamplesPerPixel",
            "<MISSING>"
        )

        number_of_frames = getattr(
            ds,
            "NumberOfFrames",
            "<MISSING>"
        )

        photometric = getattr(
            ds,
            "PhotometricInterpretation",
            "<MISSING>"
        )

        modality = getattr(
            ds,
            "Modality",
            "<MISSING>"
        )


        result.update({

            "rescale_slope_raw":
                repr(slope),

            "rescale_intercept_raw":
                repr(intercept),

            "rows_raw":
                repr(rows),

            "columns_raw":
                repr(columns),

            "bits_allocated_raw":
                repr(bits_allocated),

            "bits_stored_raw":
                repr(bits_stored),

            "high_bit_raw":
                repr(high_bit),

            "samples_per_pixel_raw":
                repr(samples_per_pixel),

            "number_of_frames_raw":
                repr(number_of_frames),

            "photometric_interpretation_raw":
                repr(photometric),

            "modality_raw":
                repr(modality)
        })


        # ----------------------------------------------------
        # Inspect raw pixel_array independently
        # ----------------------------------------------------

        try:

            pixel_array = ds.pixel_array

            result.update({

                "pixel_array_shape":
                    str(pixel_array.shape),

                "pixel_array_dtype":
                    str(pixel_array.dtype)
            })

        except Exception as pixel_error:

            result.update({

                "pixel_array_shape":
                    "ERROR",

                "pixel_array_dtype":
                    "ERROR"
            })

            print(
                "    pixel_array ERROR:",
                type(pixel_error).__name__,
                str(pixel_error)
            )


        # ----------------------------------------------------
        # MOST IMPORTANT TEST
        #
        # Call the project's existing decoder directly.
        # ----------------------------------------------------

        try:

            decoded_result = (
                dicom_to_float_image(ds)
            )

            # Depending on the existing function,
            # it may return an array or tuple.

            if isinstance(
                decoded_result,
                tuple
            ):

                decoded_array = decoded_result[0]

            else:

                decoded_array = decoded_result


            result.update({

                "dicom_to_float_image_status":
                    "SUCCESS",

                "dicom_to_float_image_error":
                    "",

                "decoded_shape":
                    str(
                        decoded_array.shape
                    )
                    if hasattr(
                        decoded_array,
                        "shape"
                    )
                    else "",

                "decoded_dtype":
                    str(
                        decoded_array.dtype
                    )
                    if hasattr(
                        decoded_array,
                        "dtype"
                    )
                    else ""
            })


            print(
                "    dicom_to_float_image: SUCCESS"
            )


        except Exception as decoder_error:

            result.update({

                "dicom_to_float_image_status":
                    "ERROR",

                "dicom_to_float_image_error":
                    (
                        f"{type(decoder_error).__name__}: "
                        f"{decoder_error}"
                    ),

                "full_traceback":
                    traceback.format_exc()
            })


            print(
                "    dicom_to_float_image: ERROR"
            )

            print(
                "    ",
                type(decoder_error).__name__,
                ":",
                decoder_error
            )


    except Exception as e:

        result.update({

            "dicom_to_float_image_status":
                "NOT_TESTED",

            "dicom_to_float_image_error":
                "DICOM inspection failed",

            "full_traceback":
                traceback.format_exc()
        })


        print(
            "    DICOM inspection ERROR:",
            type(e).__name__,
            str(e)
        )


    diagnostic_rows.append(
        result
    )


# ============================================================
# 5. Create diagnostic dataframe
# ============================================================

diagnostic_df = pd.DataFrame(
    diagnostic_rows
)


# ============================================================
# 6. Save diagnostic report
# ============================================================

diagnostic_file = (
    QC_DIR /
    "TB_Portals_March2025_Notebook06_43_Error_Diagnostic.csv"
)

diagnostic_df.to_csv(
    diagnostic_file,
    index=False
)


# ============================================================
# 7. Display important diagnostic information
# ============================================================

print()
print("=" * 80)
print("DIAGNOSTIC SUMMARY")
print("=" * 80)

print(
    "\nTotal failed records inspected:",
    len(diagnostic_df)
)


print(
    "\nDICOM existence:"
)

print(
    diagnostic_df[
        "dicom_exists"
    ].value_counts(
        dropna=False
    )
)


print(
    "\ndicom_to_float_image() status:"
)

print(
    diagnostic_df[
        "dicom_to_float_image_status"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nRecords with RescaleSlope = None:"
)

print(
    (
        diagnostic_df[
            "rescale_slope_raw"
        ].astype(str)
        == "None"
    ).sum()
)


print(
    "\nRecords with RescaleIntercept = None:"
)

print(
    (
        diagnostic_df[
            "rescale_intercept_raw"
        ].astype(str)
        == "None"
    ).sum()
)


print(
    "\nFirst diagnostic rows:"
)

display(
    diagnostic_df[
        [
            "condition_id",
            "rescale_slope_raw",
            "rescale_intercept_raw",
            "rows_raw",
            "columns_raw",
            "bits_allocated_raw",
            "bits_stored_raw",
            "number_of_frames_raw",
            "photometric_interpretation_raw",
            "dicom_to_float_image_status",
            "dicom_to_float_image_error"
        ]
    ].head(20)
)


print(
    "\nDiagnostic file:"
)

print(
    diagnostic_file
)

print("=" * 80)

DIAGNOSTIC: FAILED CXR EXTRACTION RECORDS
QC rows: 2081
Failed records: 43

Beginning detailed diagnostic...

[01/43] 01028dc8-e2c8-4d68-8294-f4422f36a9cd
    dicom_to_float_image: ERROR
     TypeError : float() argument must be a string or a real number, not 'NoneType'
[02/43] 01b78c92-c8f6-4564-b43f-a3a774f3d9eb
    dicom_to_float_image: ERROR
     TypeError : float() argument must be a string or a real number, not 'NoneType'
[03/43] 039ec7b4-66bb-4344-87fc-08b801861c75
    dicom_to_float_image: ERROR
     TypeError : float() argument must be a string or a real number, not 'NoneType'
[04/43] 06077978-9d31-4729-976e-80f8cb8d406f
    dicom_to_float_image: ERROR
     TypeError : float() argument must be a string or a real number, not 'NoneType'
[05/43] 0cdc1ea9-4ac4-4fc2-b638-cff65e68cd10
    dicom_to_float_image: ERROR
     TypeError : float() argument must be a string or a real number, not 'NoneType'
[06/43] 0e058bf2-57a7-43c3-941a-497daa2ee329
    dicom_to_float_image: ERROR
     Typ

,condition_id,rescale_slope_raw,rescale_intercept_raw,rows_raw,columns_raw,bits_allocated_raw,bits_stored_raw,number_of_frames_raw,photometric_interpretation_raw,dicom_to_float_image_status,dicom_to_float_image_error
0,01028dc8-e2c8-4d68-8294-f4422f36a9cd,None,None,3000,4000,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
1,01b78c92-c8f6-4564-b43f-a3a774f3d9eb,None,None,2140,1760,16,10,'<MISSING>','MONOCHROME1',ERROR,TypeError: float() argument must be a string o...
2,039ec7b4-66bb-4344-87fc-08b801861c75,None,None,2898,3551,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
3,06077978-9d31-4729-976e-80f8cb8d406f,None,None,2902,3404,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
4,0cdc1ea9-4ac4-4fc2-b638-cff65e68cd10,None,None,1760,2140,16,10,'<MISSING>','MONOCHROME1',ERROR,TypeError: float() argument must be a string o...
5,0e058bf2-57a7-43c3-941a-497daa2ee329,None,None,3000,4000,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
6,118770cf-6d3b-49c4-bb25-0b0e421b4a47,None,None,3000,3543,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
7,1669ff70-6aa1-44e4-bfab-c0f93dda4371,None,None,1063,1417,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
8,19655f19-d4d3-4fa8-be02-f2bb18c1780d,None,None,2944,3306,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...
9,23900749-07b4-4d13-bdc7-efcad45939d5,None,None,2932,3671,8,8,'<MISSING>','MONOCHROME2',ERROR,TypeError: float() argument must be a string o...



Diagnostic file:
C:\TBP\Metadata\Step_3B_6_51_CXR_Extraction_Technical_Preprocessing\QC\TB_Portals_March2025_Notebook06_43_Error_Diagnostic.csv


In [27]:
# ============================================================
# CELL 21B — CORRECT NONE-SAFE DICOM DECODER
# ============================================================

def safe_dicom_float(value, default):
    """
    Safely convert a DICOM numeric value to float.

    If the DICOM attribute is absent or explicitly None,
    use the specified technical default.
    """

    if value is None:
        return float(default)

    try:
        return float(value)

    except (TypeError, ValueError):
        return float(default)


def dicom_to_float_image(ds):
    """
    Decode a single-frame DICOM into a floating-point image.

    Handles:
        - RescaleSlope
        - RescaleIntercept
        - MONOCHROME1
        - MONOCHROME2
        - grayscale images

    Missing RescaleSlope:
        default = 1.0

    Missing RescaleIntercept:
        default = 0.0
    """

    # --------------------------------------------------------
    # Read pixel data
    # --------------------------------------------------------

    arr = ds.pixel_array.astype(
        np.float32
    )


    # --------------------------------------------------------
    # Read rescale parameters safely
    # --------------------------------------------------------

    slope = safe_dicom_float(
        getattr(
            ds,
            "RescaleSlope",
            None
        ),
        1.0
    )

    intercept = safe_dicom_float(
        getattr(
            ds,
            "RescaleIntercept",
            None
        ),
        0.0
    )


    # --------------------------------------------------------
    # Apply DICOM rescale transformation
    #
    # Output = Stored Pixel × Slope + Intercept
    # --------------------------------------------------------

    arr = (
        arr * slope
    ) + intercept


    # --------------------------------------------------------
    # Handle MONOCHROME1
    #
    # MONOCHROME1:
    # minimum pixel value is displayed as white.
    #
    # We invert it so that the resulting representation
    # follows the conventional MONOCHROME2 orientation.
    # --------------------------------------------------------

    photometric = str(
        getattr(
            ds,
            "PhotometricInterpretation",
            ""
        )
    ).upper()


    if photometric == "MONOCHROME1":

        finite_mask = np.isfinite(
            arr
        )

        if finite_mask.any():

            finite_values = arr[
                finite_mask
            ]

            arr_min = finite_values.min()
            arr_max = finite_values.max()

            arr = (
                arr_max
                + arr_min
                - arr
            )


    # --------------------------------------------------------
    # Validate output
    # --------------------------------------------------------

    if arr.size == 0:

        raise ValueError(
            "Decoded DICOM image is empty."
        )


    if not np.isfinite(arr).any():

        raise ValueError(
            "Decoded DICOM image contains no finite values."
        )


    return arr, photometric

In [29]:
# ============================================================
# CELL 21C — VALIDATE THE FIX ON THE 43 FAILED DICOMS
# ============================================================

print("=" * 80)
print("VALIDATING CORRECTED DICOM DECODER")
print("=" * 80)


# ============================================================
# 1. Load diagnostic report
# ============================================================

diagnostic_file = (
    QC_DIR /
    "TB_Portals_March2025_Notebook06_43_Error_Diagnostic.csv"
)

diagnostic_df = pd.read_csv(
    diagnostic_file
)


# ============================================================
# 2. Select DICOM files that actually exist
#
# IMPORTANT:
# Parentheses are required around the comparison.
# ============================================================

failed_paths = diagnostic_df[
    diagnostic_df["dicom_exists"] == True
]["dicom_path"].tolist()


print(
    "DICOM files to test:",
    len(failed_paths)
)

print()


# ============================================================
# 3. Validate each failed DICOM
# ============================================================

validation_rows = []


for position, dicom_path_str in enumerate(
    failed_paths,
    start=1
):

    dicom_path = Path(
        dicom_path_str
    )

    print(
        f"[{position:02d}/{len(failed_paths)}] "
        f"{dicom_path.name}"
    )


    result = {

        "dicom_path":
            str(dicom_path),

        "status":
            "ERROR",

        "shape":
            None,

        "dtype":
            None,

        "min":
            None,

        "max":
            None,

        "mean":
            None,

        "photometric":
            None,

        "error":
            None
    }


    try:

        # ----------------------------------------------------
        # Read DICOM
        # ----------------------------------------------------

        ds = pydicom.dcmread(
            dicom_path,
            force=False
        )


        # ----------------------------------------------------
        # Decode using corrected decoder
        # ----------------------------------------------------

        arr, photometric = (
            dicom_to_float_image(ds)
        )


        # ----------------------------------------------------
        # Validate decoded image
        # ----------------------------------------------------

        if arr.ndim != 2:

            raise ValueError(
                f"Expected 2D image, got shape {arr.shape}"
            )


        if arr.size == 0:

            raise ValueError(
                "Image contains zero pixels."
            )


        if not np.isfinite(arr).all():

            raise ValueError(
                "Image contains NaN or infinite values."
            )


        # ----------------------------------------------------
        # Record successful validation
        # ----------------------------------------------------

        result.update({

            "status":
                "PASS",

            "shape":
                str(arr.shape),

            "dtype":
                str(arr.dtype),

            "min":
                float(arr.min()),

            "max":
                float(arr.max()),

            "mean":
                float(arr.mean()),

            "photometric":
                photometric
        })


        print(
            "    -> PASS",
            arr.shape,
            arr.dtype
        )


    except Exception as e:

        result.update({

            "status":
                "ERROR",

            "error":
                f"{type(e).__name__}: {e}"
        })


        print(
            "    -> ERROR:",
            type(e).__name__,
            str(e)
        )


    validation_rows.append(
        result
    )


# ============================================================
# 4. Create validation DataFrame
# ============================================================

decoder_validation_df = pd.DataFrame(
    validation_rows
)


# ============================================================
# 5. Save validation report
# ============================================================

decoder_validation_file = (
    QC_DIR /
    "TB_Portals_March2025_Notebook06_43_DICOM_Decoder_Validation.csv"
)

decoder_validation_df.to_csv(
    decoder_validation_file,
    index=False
)


# ============================================================
# 6. Final summary
# ============================================================

print()
print("=" * 80)
print("DECODER VALIDATION SUMMARY")
print("=" * 80)

status_counts = (
    decoder_validation_df[
        "status"
    ].value_counts(
        dropna=False
    )
)

print(
    status_counts
)


pass_count = (
    decoder_validation_df[
        "status"
    ] == "PASS"
).sum()


error_count = (
    decoder_validation_df[
        "status"
    ] == "ERROR"
).sum()


print()
print(
    "Expected:",
    len(failed_paths),
    "records"
)

print(
    "PASS:",
    pass_count
)

print(
    "ERROR:",
    error_count
)


print()
print(
    "Validation file:"
)

print(
    decoder_validation_file
)

print("=" * 80)

VALIDATING CORRECTED DICOM DECODER
DICOM files to test: 43

[01/43] 01028dc8-e2c8-4d68-8294-f4422f36a9cd_2b07c9c94489.dcm
    -> PASS (3000, 4000) float32
[02/43] 01b78c92-c8f6-4564-b43f-a3a774f3d9eb_16feb740dd44.dcm
    -> PASS (2140, 1760) float32
[03/43] 039ec7b4-66bb-4344-87fc-08b801861c75_528ff347e49a.dcm
    -> PASS (2898, 3551) float32
[04/43] 06077978-9d31-4729-976e-80f8cb8d406f_521bec7d23b4.dcm
    -> PASS (2902, 3404) float32
[05/43] 0cdc1ea9-4ac4-4fc2-b638-cff65e68cd10_2ca30b085dfb.dcm
    -> PASS (1760, 2140) float32
[06/43] 0e058bf2-57a7-43c3-941a-497daa2ee329_b074d98ac91b.dcm
    -> PASS (3000, 4000) float32
[07/43] 118770cf-6d3b-49c4-bb25-0b0e421b4a47_9939d201af6a.dcm
    -> PASS (3000, 3543) float32
[08/43] 1669ff70-6aa1-44e4-bfab-c0f93dda4371_0775479abfb0.dcm
    -> PASS (1063, 1417) float32
[09/43] 19655f19-d4d3-4fa8-be02-f2bb18c1780d_fe98651eaefc.dcm
    -> PASS (2944, 3306) float32
[10/43] 23900749-07b4-4d13-bdc7-efcad45939d5_bb9e5a3147ce.dcm
    -> PASS (2932, 3671

In [32]:
# ============================================================
# CELL 22 — FINAL CXR EXTRACTION QUALITY GATE
# RESUMABLE-EXTRACTION AWARE VERSION
# ============================================================

print("=" * 80)
print("CELL 22 — FINAL CXR EXTRACTION QUALITY GATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. EXPECTED COHORT SIZE
# ------------------------------------------------------------

EXPECTED = 2081

print()
print("Expected records:", EXPECTED)


# ------------------------------------------------------------
# 2. LOCATE QC TABLE
# ------------------------------------------------------------

QC_FILE = (
    QC_DIR /
    "TB_Portals_March2025_Notebook06_CXR_Technical_QC.csv"
)

if not QC_FILE.exists():
    raise FileNotFoundError(
        f"Technical QC file not found:\n{QC_FILE}"
    )

qc_df = pd.read_csv(QC_FILE)

print("QC file:", QC_FILE)
print("QC rows:", len(qc_df))


# ------------------------------------------------------------
# 3. QC TABLE STRUCTURE
# ------------------------------------------------------------

required_qc_columns = {
    "condition_id",
    "status"
}

missing_qc_columns = (
    required_qc_columns -
    set(qc_df.columns)
)

if missing_qc_columns:
    raise RuntimeError(
        "Required QC columns are missing:\n"
        + "\n".join(
            sorted(missing_qc_columns)
        )
    )

print()
print("QC schema: PASS")


# ------------------------------------------------------------
# 4. QC ROW COUNT
# ------------------------------------------------------------

if len(qc_df) != EXPECTED:

    raise RuntimeError(
        f"QC table contains {len(qc_df)} rows; "
        f"expected exactly {EXPECTED}."
    )

print("QC row count: PASS")


# ------------------------------------------------------------
# 5. CONDITION-ID UNIQUENESS
# ------------------------------------------------------------

duplicate_conditions = (
    qc_df["condition_id"]
    .astype(str)
    .duplicated()
    .sum()
)

print()
print(
    "Duplicate condition IDs:",
    duplicate_conditions
)

if duplicate_conditions != 0:

    raise RuntimeError(
        "Duplicate condition_id values detected "
        "in the technical QC table."
    )

print("Condition-level uniqueness: PASS")


# ------------------------------------------------------------
# 6. STATUS DISTRIBUTION
# ------------------------------------------------------------

status_counts = (
    qc_df["status"]
    .value_counts(dropna=False)
)

print()
print("QC status distribution:")
print(status_counts)


# ------------------------------------------------------------
# 7. RESUMABLE EXTRACTION STATUS LOGIC
# ------------------------------------------------------------
#
# Cell 21 is intentionally resumable.
#
# Therefore:
#
#     PASS   = newly processed successfully
#     REUSED = previously processed successfully
#
# Both represent successful technical extraction.
#
# Any other status is considered a failure.
# ------------------------------------------------------------

VALID_SUCCESS_STATUSES = {
    "PASS",
    "REUSED"
}

success_mask = (
    qc_df["status"]
    .isin(VALID_SUCCESS_STATUSES)
)

passed = int(
    success_mask.sum()
)

failed = int(
    (~success_mask).sum()
)

print()
print(
    "Successful records:",
    passed
)

print(
    "Failed records:",
    failed
)

print()
print(
    "Accepted success statuses:",
    sorted(VALID_SUCCESS_STATUSES)
)


# ------------------------------------------------------------
# 8. FAILURE REPORT IF NECESSARY
# ------------------------------------------------------------

if passed != EXPECTED:

    failures = qc_df[
        ~success_mask
    ].copy()

    failure_file = (
        QC_DIR /
        "Notebook06_CXR_Extraction_Failures.csv"
    )

    failures.to_csv(
        failure_file,
        index=False
    )

    print()
    print(
        "Failure report:",
        failure_file
    )

    raise RuntimeError(
        f"{failed} of {EXPECTED} selected CXRs "
        "have unsuccessful extraction status."
    )

print()
print(
    "Technical extraction status: PASS"
)


# ------------------------------------------------------------
# 9. CHECK DICOM OUTPUT FILES
# ------------------------------------------------------------

dicom_files = list(
    EXTRACTED_DIR.glob("*.dcm")
)

dicom_count = len(dicom_files)

print()
print(
    "DICOM files present:",
    dicom_count
)

if dicom_count != EXPECTED:

    raise RuntimeError(
        f"Expected {EXPECTED} DICOM files, "
        f"but found {dicom_count}."
    )

print("DICOM file count: PASS")


# ------------------------------------------------------------
# 10. CHECK PNG OUTPUT FILES
# ------------------------------------------------------------

png_files = list(
    PNG_DIR.glob("*.png")
)

png_count = len(png_files)

print(
    "PNG files present:",
    png_count
)

if png_count != EXPECTED:

    raise RuntimeError(
        f"Expected {EXPECTED} PNG files, "
        f"but found {png_count}."
    )

print("PNG file count: PASS")


# ------------------------------------------------------------
# 11. CHECK UNIQUE OUTPUT FILENAMES
# ------------------------------------------------------------

dicom_stems = {
    p.stem
    for p in dicom_files
}

png_stems = {
    p.stem
    for p in png_files
}

print()
print(
    "Unique DICOM stems:",
    len(dicom_stems)
)

print(
    "Unique PNG stems:",
    len(png_stems)
)

if len(dicom_stems) != EXPECTED:

    raise RuntimeError(
        "Duplicate DICOM output stems detected."
    )

if len(png_stems) != EXPECTED:

    raise RuntimeError(
        "Duplicate PNG output stems detected."
    )

print("Output filename uniqueness: PASS")


# ------------------------------------------------------------
# 12. DICOM ↔ PNG CORRESPONDENCE
# ------------------------------------------------------------

missing_png = sorted(
    dicom_stems - png_stems
)

missing_dicom = sorted(
    png_stems - dicom_stems
)

print()
print(
    "DICOM without PNG:",
    len(missing_png)
)

print(
    "PNG without DICOM:",
    len(missing_dicom)
)

if missing_png:

    print(
        "Example missing PNG stems:",
        missing_png[:10]
    )

    raise RuntimeError(
        "Some DICOM outputs do not have "
        "corresponding PNG outputs."
    )

if missing_dicom:

    print(
        "Example missing DICOM stems:",
        missing_dicom[:10]
    )

    raise RuntimeError(
        "Some PNG outputs do not have "
        "corresponding DICOM outputs."
    )

print("DICOM ↔ PNG correspondence: PASS")


# ------------------------------------------------------------
# 13. LOCKED COHORT ↔ QC CONDITION INTEGRITY
# ------------------------------------------------------------

if CONDITION_COL not in cohort.columns:

    raise RuntimeError(
        f"Locked cohort does not contain "
        f"'{CONDITION_COL}'."
    )

cohort_conditions = set(
    cohort[CONDITION_COL]
    .astype(str)
    .str.strip()
)

qc_conditions = set(
    qc_df["condition_id"]
    .astype(str)
    .str.strip()
)

missing_from_qc = (
    cohort_conditions -
    qc_conditions
)

extra_in_qc = (
    qc_conditions -
    cohort_conditions
)

print()
print(
    "Locked cohort conditions:",
    len(cohort_conditions)
)

print(
    "QC conditions:",
    len(qc_conditions)
)

print(
    "Cohort conditions missing from QC:",
    len(missing_from_qc)
)

print(
    "Extra QC conditions:",
    len(extra_in_qc)
)

if missing_from_qc:

    raise RuntimeError(
        "Some locked cohort conditions are "
        "missing from the QC table."
    )

if extra_in_qc:

    raise RuntimeError(
        "QC contains condition IDs that are "
        "not present in the locked cohort."
    )

print(
    "Cohort ↔ QC condition integrity: PASS"
)


# ------------------------------------------------------------
# 14. PNG READABILITY
# ------------------------------------------------------------

try:
    from PIL import Image

except ImportError as e:

    raise ImportError(
        "Pillow is required for PNG readability validation."
    ) from e


png_read_errors = []

print()
print("Validating PNG readability...")

for png_path in sorted(png_files):

    try:

        with Image.open(png_path) as img:

            img.verify()

    except Exception as e:

        png_read_errors.append({
            "png_path": str(png_path),
            "error": f"{type(e).__name__}: {e}"
        })


print(
    "PNG readability errors:",
    len(png_read_errors)
)

if png_read_errors:

    png_error_df = pd.DataFrame(
        png_read_errors
    )

    png_error_file = (
        QC_DIR /
        "Notebook06_PNG_Readability_Failures.csv"
    )

    png_error_df.to_csv(
        png_error_file,
        index=False
    )

    raise RuntimeError(
        f"{len(png_read_errors)} PNG files "
        "failed readability validation.\n"
        f"Failure report:\n{png_error_file}"
    )

print(
    "PNG readability: PASS"
)


# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------

print()
print("=" * 80)
print("CELL 22 — QUALITY GATE SUMMARY")
print("=" * 80)

print()
print("Expected records        :", EXPECTED)
print("QC records              :", len(qc_df))
print("Successful QC records   :", passed)
print("Failed QC records       :", failed)
print("DICOM files             :", dicom_count)
print("PNG files               :", png_count)
print("Unique DICOM stems      :", len(dicom_stems))
print("Unique PNG stems        :", len(png_stems))
print("PNG readability errors  :", len(png_read_errors))
print(
    "Cohort ↔ QC mismatches  :",
    len(missing_from_qc) + len(extra_in_qc)
)

print()
print("SUCCESS STATUS BREAKDOWN:")
print(
    qc_df["status"]
    .value_counts()
)

print()
print("ALL 2,081 SELECTED CXRs PASSED THE")
print("FINAL TECHNICAL EXTRACTION QUALITY GATE.")

print()
print("Extraction quality gate: PASS")

print("=" * 80)

CELL 22 — FINAL CXR EXTRACTION QUALITY GATE

Expected records: 2081
QC file: C:\TBP\Metadata\Step_3B_6_51_CXR_Extraction_Technical_Preprocessing\QC\TB_Portals_March2025_Notebook06_CXR_Technical_QC.csv
QC rows: 2081

QC schema: PASS
QC row count: PASS

Duplicate condition IDs: 0
Condition-level uniqueness: PASS

QC status distribution:
status
REUSED    2038
PASS        43
Name: count, dtype: int64

Successful records: 2081
Failed records: 0

Accepted success statuses: ['PASS', 'REUSED']

Technical extraction status: PASS

DICOM files present: 2081
DICOM file count: PASS
PNG files present: 2081
PNG file count: PASS

Unique DICOM stems: 2081
Unique PNG stems: 2081
Output filename uniqueness: PASS

DICOM without PNG: 0
PNG without DICOM: 0
DICOM ↔ PNG correspondence: PASS

Locked cohort conditions: 2081
QC conditions: 2081
Cohort conditions missing from QC: 0
Extra QC conditions: 0
Cohort ↔ QC condition integrity: PASS

Validating PNG readability...
PNG readability errors: 0
PNG readabilit